# Демонстрація можливостей tokamld (PyTorch & Fusion ML)

Цей ноутбук демонструє роботу відкритих інструментів `tokamld` на **25 навчальних розрядах** DIII-D (#000–#024):
1. **`tokamld.gs`**: Диференційовний оператор $\Delta^*\psi$, неперервний шлюз $G_{GS}$ та нев'язка $g(\psi)$;
2. **`tokamld.topology`**: Канонічна топологія (1 O-точка, 0 X-точок всередині LCFS);
3. **`tokamld.conformal`**: Одночасні конформні смуги (Simultaneous Conformal Bands, C6);
4. **`tokamld.fieldline`**: Інтегратор силових ліній, функція Гріна (E33) та формула ширини острова (VR15).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch

sys.path.insert(0, "../src")
sys.path.insert(0, "../fusion equilibrium challenge/starter")
sys.path.insert(0, "../fusion equilibrium challenge/starter/fusion_scoring")
sys.path.insert(0, "../Our try/03-deep-dives/D1-psi-to-scalars/c2")

import tokamld
from tokamld.gs import delta_star, residual, gs_inconsistency, gate
from tokamld.topology import check_canonical_topology, inside_lcfs
from tokamld.conformal import SimultaneousConformalBands
from tokamld.fieldline import green_function, magnetic_island_width
import train as c2_train

# 1. Завантаження 25 розрядів
shots_25 = [c2_train.load([i])[0] for i in range(25)]
print(f"Loaded {len(shots_25)} training shots.")

In [ ]:
# 2. Перевірка нев'язки Град-Шафранова та чутливості до 1% шуму на 25 розрядах
mask = np.load("../fusion equilibrium challenge/starter/fusion_scoring/masks/d3d_envelope.npz")
R, Z = mask["grid_R"], mask["grid_Z"]
mc, mf = mask["mask_coarse"].astype(bool), mask["mask_coarse"].astype(np.float64)

g_true, g_noise = [], []
rng = np.random.default_rng(42)
for s in shots_25:
    psi = s["psi"][len(s["psi"]) // 2].astype(np.float64)
    psi_n = psi + 0.01 * np.std(psi) * rng.standard_normal(psi.shape)
    gt, _ = gs_inconsistency(psi, R, Z, mc, mf)
    gn, _ = gs_inconsistency(psi_n, R, Z, mc, mf)
    if np.isfinite(gt) and np.isfinite(gn):
        g_true.append(gt)
        g_noise.append(gn)

print(f"Median g(true):       {np.median(g_true):.4f}")
print(f"Median g(true + 1%):  {np.median(g_noise):.4f} (E6/E38 benchmark ~ 0.63)")

In [ ]:
# 3. Одночасні конформні смуги (Conformal UQ)
# 20 розрядів на калібрування, 5 на тест
y_cal = np.concatenate([s["psi"][:5] for s in shots_25[:20]])
y_cal_noisy = y_cal + np.random.normal(0, 0.02, size=y_cal.shape)
y_test = np.concatenate([s["psi"][:5] for s in shots_25[20:]])
y_test_noisy = y_test + np.random.normal(0, 0.02, size=y_test.shape)

cb = SimultaneousConformalBands(alpha=0.90)
cb.fit(y_cal, y_cal_noisy, mask=mc)
cov = cb.evaluate_coverage(y_test, y_test_noisy, mask=mc)
print("Conformal coverage on held-out shots:", cov)

In [ ]:
# 4. Регресійні фізичні формули (E33, VR15)
psi_loop = green_function(1.5, 0.25, 1.0, 0.0)
w_island = magnetic_island_width(psi_tilde=1e-3, q_res=2.0, dq_dpsi=5.0)
print(f"Green function mutual flux: {psi_loop:.6e} Wb/rad/A")
print(f"Magnetic island width W:     {w_island:.6f} (exact = 0.080000)")